<a href="https://colab.research.google.com/github/Arnav2903/fraud-risk-monitoring-dashboard/blob/main/fraud_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
df=pd.read_csv("creditcard.csv")
df.head()
df.shape

(284807, 31)

In [ ]:
np.random.seed(42)

num_customers = 10000
customer_ids = np.random.randint(1, num_customers+1, size=len(df))

df['customer_id'] = customer_ids

In [ ]:
df['transaction_hour'] = (df['Time'] // 3600) % 24
device_types = ['Mobile', 'Web', 'POS']
df['device_type'] = np.random.choice(device_types, size=len(df), p=[0.6, 0.3, 0.1])

cities = ['Delhi', 'Mumbai', 'Bangalore', 'Chandigarh', 'Hyderabad', 'Pune']
df['city'] = np.random.choice(cities, size=len(df))
txn_types = ['UPI', 'Credit Card', 'Wallet']
df['transaction_type'] = np.random.choice(txn_types, size=len(df), p=[0.5, 0.3, 0.2])


In [ ]:
customer_tenure = pd.DataFrame({
    'customer_id': range(1, num_customers+1),
    'customer_tenure_days': np.random.randint(1, 1000, num_customers)
})

df = df.merge(customer_tenure, on='customer_id', how='left')
threshold = df['Amount'].quantile(0.75)
df['is_high_amount'] = (df['Amount'] > threshold).astype(int)
df.rename(columns={'Class': 'is_fraud'}, inplace=True)


In [ ]:
dashboard_df = df[[
    'customer_id',
    'Amount',
    'transaction_hour',
    'device_type',
    'city',
    'transaction_type',
    'customer_tenure_days',
    'is_high_amount',
    'is_fraud'
]]

dashboard_df.head()

,customer_id,Amount,transaction_hour,device_type,city,transaction_type,customer_tenure_days,is_high_amount,is_fraud
0,7271,149.62,0.0,Mobile,Bangalore,Credit Card,743,1,0
1,861,2.69,0.0,POS,Bangalore,UPI,140,0,0
2,5391,378.66,0.0,Web,Chandigarh,UPI,997,1,0
3,5192,123.50,0.0,Mobile,Hyderabad,UPI,757,1,0
4,5735,69.99,0.0,Web,Chandigarh,UPI,332,0,0


In [ ]:
df.to_csv("fraud_model_data.csv", index=False)
dashboard_df.to_csv("fraud_dashboard_data.csv", index=False)

In [ ]:
df['is_fraud'].value_counts()

,count
is_fraud,
0,284315
1,492


In [ ]:
#regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.utils import resample
import pandas as pd

df = pd.read_csv("fraud_model_data.csv")

# Encode categorical variables
df['device_type'] = LabelEncoder().fit_transform(df['device_type'])
df['city'] = LabelEncoder().fit_transform(df['city'])
df['transaction_type'] = LabelEncoder().fit_transform(df['transaction_type'])

# Features
X = df[['Amount','transaction_hour','device_type','customer_tenure_days','is_high_amount']]
y = df['is_fraud']

# Handle imbalance using class_weight
model = LogisticRegression(max_iter=1000, class_weight='balanced')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model.fit(X_train, y_train)

y_pred_prob = model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, y_pred_prob)
print("AUC Score:", auc)

AUC Score: 0.5991034371360464


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import pandas as pd

df = pd.read_csv("fraud_model_data.csv")

# Encode categorical variables
df['device_type'] = LabelEncoder().fit_transform(df['device_type'])
df['city'] = LabelEncoder().fit_transform(df['city'])
df['transaction_type'] = LabelEncoder().fit_transform(df['transaction_type'])

# Select features (include V1–V28)
feature_cols = [col for col in df.columns if col.startswith('V')] + \
               ['Amount','transaction_hour','device_type','customer_tenure_days','is_high_amount']

X = df[feature_cols]
y = df['is_fraud']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

# Train model
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

# Evaluate
y_pred_prob = model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_pred_prob)

print("Improved AUC Score:", auc)

Improved AUC Score: 0.9825952580265465


In [ ]:
# Generate probabilities on full dataset
df['fraud_probability'] = model.predict_proba(X_scaled)[:,1]

# Create risk score (0–100)
df['risk_score'] = (df['fraud_probability'] * 100).round(2)

# Create risk segments
df['risk_segment'] = pd.cut(
    df['risk_score'],
    bins=[0,30,70,100],
    labels=['Low','Medium','High']
)

In [ ]:
df['risk_segment'].value_counts()

,count
risk_segment,
Low,258899
Medium,14456
High,3776


In [ ]:
df.groupby('risk_segment')['is_fraud'].mean()

/tmp/ipython-input-368063666.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('risk_segment')['is_fraud'].mean()


,is_fraud
risk_segment,
Low,0.000089
Medium,0.001660
High,0.117850


In [ ]:
df = df.sort_values(by='risk_score', ascending=False)

top_5_percent = int(len(df) * 0.05)

blocked_transactions = df.head(top_5_percent)

prevented_loss = blocked_transactions[blocked_transactions['is_fraud']==1]['Amount'].sum()

print("Potential Fraud Prevented:", prevented_loss)

Potential Fraud Prevented: 53773.520000000004


In [ ]:
final_dashboard = df[[
    'customer_id',
    'Amount',
    'transaction_hour',
    'device_type',
    'city',
    'transaction_type',
    'customer_tenure_days',
    'is_high_amount',
    'is_fraud',
    'risk_score',
    'risk_segment'
]]

final_dashboard.to_csv("fraud_final_dashboard_data.csv", index=False)

In [ ]:
print(LabelEncoder().fit(['Mobile','POS','Web']).classes_)

['Mobile' 'POS' 'Web']
